# Lezione 54 — Il classificatore del tipo di memoria

> **Modulo:** capstone · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 15 (bag of words), Lezione 9 (softmax/loss).
>
> Obiettivo pratico unico: addestrare in NumPy un classificatore
> **bag-of-words + regressione softmax** che assegna a una memoria il suo tipo,
> riproducendo il salto di qualita' della Lezione 15.

## Teoria essenziale

Il primo componente reale: dato il testo, prevedere `type` ∈ {episodic, semantic,
preference}. Come nella Lezione 15, rappresentiamo il testo come **bag of words**
(vocabolario costruito **solo sul train**, per non fare leakage) e usiamo una
**regressione softmax** (Lezione 9) addestrata a mano in NumPy.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

proc = Path('..') / 'datasets' / 'processed'
TIPI = ["episodic", "semantic", "preference"]
idx_tipo = {t: i for i, t in enumerate(TIPI)}

def carica(nome):
    df = pd.read_csv(proc / nome)
    return df[df['type'].isin(TIPI)].reset_index(drop=True)

train, val = carica('memory_train.csv'), carica('memory_val.csv')

def tok(t):
    return re.findall(r"[a-zA-Z]+", str(t).lower())

# vocabolario SOLO dal train
vocab = {}
for t in train['text']:
    for w in tok(t):
        if w not in vocab:
            vocab[w] = len(vocab)
print("dimensione vocabolario (solo train):", len(vocab))

def bow(testo):
    v = np.zeros(len(vocab))
    for w in tok(testo):
        if w in vocab:
            v[vocab[w]] += 1.0
    return v

Xtr = np.vstack([bow(t) for t in train['text']])
ytr = np.array([idx_tipo[t] for t in train['type']])
Xva = np.vstack([bow(t) for t in val['text']])
yva = np.array([idx_tipo[t] for t in val['type']])
print("X train:", Xtr.shape, "| X val:", Xva.shape)

dimensione vocabolario (solo train): 64
X train: (212, 64) | X val: (19, 64)


In [2]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

def accuratezza(W, b, X, y):
    return float((softmax(X @ W + b).argmax(axis=1) == y).mean())

rng = np.random.default_rng(54)
K = len(TIPI)
W = np.zeros((len(vocab), K))
b = np.zeros(K)
lr, lam = 0.5, 1e-3
Ytr = np.eye(K)[ytr]

# maggioranza come baseline onesta
maggioranza = np.bincount(ytr).argmax()
print(f"baseline maggioranza (val): {(yva == maggioranza).mean():.2f}")
for passo in range(300):
    P = softmax(Xtr @ W + b)
    gW = Xtr.T @ (P - Ytr) / len(Xtr) + lam * W
    gb = (P - Ytr).mean(axis=0)
    W -= lr * gW
    b -= lr * gb
    if passo % 100 == 0 or passo == 299:
        print(f"passo {passo:3d}: acc train {accuratezza(W,b,Xtr,ytr):.2f}  "
              f"val {accuratezza(W,b,Xva,yva):.2f}")

baseline maggioranza (val): 0.68
passo   0: acc train 0.80  val 0.74
passo 100: acc train 1.00  val 1.00
passo 200: acc train 1.00  val 1.00
passo 299: acc train 1.00  val 1.00


Leggi le accuratezze: leggere il testo (bag of words) porta il classificatore
ben sopra la baseline di maggioranza — lo stesso effetto della Lezione 15. Il
divario train/val dice quanto generalizza.

In [3]:
acc_val = accuratezza(W, b, Xva, yva)
def classifica_tipo(testo):
    return TIPI[int(softmax(bow(testo)[None, :] @ W + b).argmax())]

# controlli di non-regressione
assert acc_val > (yva == maggioranza).mean()      # batte la maggioranza
assert classifica_tipo("The user prefers morning sessions.") in TIPI
print(f"accuratezza validazione: {acc_val:.2f}")
for t in ["The user prefers tea.", "Marco went to Rome yesterday.", "Water boils at 100C."]:
    print(f"  {classifica_tipo(t):11} <- {t}")

accuratezza validazione: 1.00
  preference  <- The user prefers tea.
  episodic    <- Marco went to Rome yesterday.
  semantic    <- Water boils at 100C.


## Riepilogo (max 8 punti)

1. Primo componente: testo → `type`.
2. **Bag of words** con vocabolario **solo dal train** (no leakage).
3. **Regressione softmax** in NumPy (Lezione 9).
4. Baseline onesta: la classe di maggioranza.
5. Leggere il testo supera nettamente la maggioranza (come Lezione 15).
6. Il divario train/val misura la generalizzazione.
7. `classifica_tipo` diventa il componente della pipeline (passo 54).
8. Regolarizzazione L2 per contenere l'overfitting.

## Quiz

1. Perche' il vocabolario si costruisce solo dal train?
2. A cosa serve la baseline di maggioranza?
3. Cosa indica un ampio divario tra accuratezza train e val?

*(Risposte: 1. per evitare leakage: usare parole del val/test darebbe un
vantaggio irreale; 2. un metro minimo: il modello deve batterla per essere utile;
3. overfitting — il modello memorizza il train e generalizza male.)*